<a href="https://colab.research.google.com/github/RicardoTL75/Module_5_AI/blob/main/M5_2_Autoencoder_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
from torchinfo import summary
torch.manual_seed(0)

NUM_CLASSES = 5
weights = ResNet18_Weights.DEFAULT

# Load the pretrained source model.
# If the weights are not cached, torchvision may download them once.
try:
    model = resnet18(weights=weights)
    pretrained_available = True
    print("Loaded pretrained ResNet-18 weights.")
except Exception as e:
    # The structural parts of the exercise still run, but this fallback is NOT transfer learning.
    print("Pretrained weights unavailable:", type(e).__name__)
    print("Using random weights only so the architecture exercises can still run.")
    model = resnet18(weights=None)
    pretrained_available = False

# ResNet-18 was pretrained with an ImageNet classification head.
# Replace only the task-specific head; this new layer starts from random initialization.
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)

summary(model, input_size=(1, 3, 224, 224), col_names=["input_size", "output_size", "num_params", "trainable"])

print("Feature dimension entering the new head:", in_features)
print("Target classes:", NUM_CLASSES)
print("True transfer learning:", pretrained_available)

Loaded pretrained ResNet-18 weights.
Feature dimension entering the new head: 512
Target classes: 5
True transfer learning: True


In [ ]:
def compression_factor(depth, in_channels, latent_channels, spatial_dims):
    # each axis shrinks by axis_reduction (e.g. depth=2 -> axis_reduction=4)
    axis_reduction = 2 ** depth
    # spatial_dims axes shrink at the same time, so the element count compounds: axis_reduction**spatial_dims
    spatial_factor = axis_reduction ** spatial_dims
    channel_factor = in_channels / latent_channels
    return axis_reduction, spatial_factor, channel_factor, spatial_factor * channel_factor


def print_compression(name, depth, in_channels, latent_channels, spatial_dims):
    """Print the spatial vs channel breakdown; torchinfo.summary already covers shapes and params."""
    axis, spatial, channel, total = compression_factor(depth, in_channels, latent_channels, spatial_dims)
    print(f"{name} compression factor (depth={depth}): axis /{axis:.0f} -> spatial x{spatial:.2f} (elements) * channels x{channel:.2f} = x{total:.2f}")


def show_architecture(ae, x):
    """Print the full layer-by-layer architecture, input/output shapes and parameter counts."""
    _ = summary(ae, input_data=x, col_names=("input_size", "output_size", "num_params"), verbose=1, depth=3)

In [ ]:
class DownBlock1D(nn.Module):
    """MLP encoder step: merge pairs of samples, halving the length."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(2 * in_channels, out_channels),
            nn.ReLU(inplace=True),
            nn.LayerNorm(out_channels),
        )

    def forward(self, x):
        # x: [batch, channels, length]
        batch, channels, length = x.shape
        if length % 2 != 0:
            raise ValueError("DownBlock1D expects an even length.")
        x = x.transpose(1, 2)                    # [batch, length, channels]
        x = x.reshape(batch, length // 2, 2 * channels)
        x = self.block(x)                        # [batch, length // 2, out_channels]
        return x.transpose(1, 2)                 # [batch, out_channels, length // 2]


class UpBlock1D(nn.Module):
    """MLP decoder step: split each sample into two, doubling the length."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.out_channels = out_channels
        self.proj = nn.Linear(in_channels, 2 * out_channels)
        self.act = nn.ReLU(inplace=True)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        # x: [batch, channels, length]
        batch, _, length = x.shape
        x = x.transpose(1, 2)                    # [batch, length, channels]
        x = self.act(self.proj(x))               # [batch, length, 2 * out_channels]
        x = x.reshape(batch, length * 2, self.out_channels)
        x = self.norm(x)
        return x.transpose(1, 2)                 # [batch, out_channels, length * 2]

In [ ]:
class Autoencoder1D(nn.Module):
    """Stacks `depth` DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1):
        super().__init__()
        self.depth = depth
        channels = [in_channels] + [latent_channels] * depth
        self.encoder = nn.ModuleList(
            [DownBlock1D(channels[i], channels[i + 1]) for i in range(depth)]
        )
        self.decoder = nn.ModuleList(
            [UpBlock1D(channels[i + 1], channels[i]) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))


x1 = torch.randn(4, 1, 1024)

ae1 = Autoencoder1D(in_channels=1, latent_channels=32, depth=3)
show_architecture(ae1, x1)
print_compression("1D", depth=1, in_channels=3, latent_channels=2, spatial_dims=1)

ae1_deep = Autoencoder1D(in_channels=1, latent_channels=2, depth=2)
show_architecture(ae1_deep, x1)
print_compression("1D (depth=2)", depth=2, in_channels=3, latent_channels=2, spatial_dims=1)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
Autoencoder1D                            [4, 1, 1024]              [4, 1, 1024]              --
├─ModuleList: 1-1                        --                        --                        --
│    └─DownBlock1D: 2-1                  [4, 1, 1024]              [4, 32, 512]              --
│    │    └─Sequential: 3-1              [4, 512, 2]               [4, 512, 32]              160
│    └─DownBlock1D: 2-2                  [4, 32, 512]              [4, 32, 256]              --
│    │    └─Sequential: 3-2              [4, 256, 64]              [4, 256, 32]              2,144
│    └─DownBlock1D: 2-3                  [4, 32, 256]              [4, 32, 128]              --
│    │    └─Sequential: 3-3              [4, 128, 64]              [4, 128, 32]              2,144
├─ModuleList: 1-2                        --                        --                        --
│    └─UpBlock1D: 2-4       

In [ ]:
class ConvBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
        )
    def forward(self, x):
        return self.block(x)


class DownBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()   # <- corregir aquí
        self.conv = ConvBlock2D(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    def forward(self, x):
        return self.pool(self.conv(x))


class UpBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels, mode="convtranspose"):
        super().__init__()
        if mode == "convtranspose":
            self.upsample = nn.ConvTranspose2d(in_channels, in_channels, kernel_size=2, stride=2)
        elif mode == "linear":
            self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        else:
            raise ValueError(f"Unknown upsampling mode: {mode}")
        self.conv = ConvBlock2D(in_channels, out_channels)
    def forward(self, x):
        return self.conv(self.upsample(x))

In [ ]:
class Autoencoder2D(nn.Module):
    """Stacks depth DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1, up_mode="convtranspose"):
        super().__init__()
        self.depth = depth
        channels = [in_channels] + [latent_channels] * depth
        self.encoder = nn.ModuleList(
            [DownBlock2D(channels[i], channels[i + 1]) for i in range(depth)]
        )
        self.decoder = nn.ModuleList(
            [UpBlock2D(channels[i + 1], channels[i], mode=up_mode) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))


x2 = torch.randn(2, 1, 28, 28)

ae2 = Autoencoder2D(in_channels=1, latent_channels=4, depth=1)
show_architecture(ae2, x2)
print_compression("2D", depth=1, in_channels=1, latent_channels=4, spatial_dims=2)

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
Autoencoder2D                                 [2, 1, 28, 28]            [2, 1, 28, 28]            --
├─ModuleList: 1-1                             --                        --                        --
│    └─DownBlock2D: 2-1                       [2, 1, 28, 28]            [2, 4, 14, 14]            --
│    │    └─ConvBlock2D: 3-1                  [2, 1, 28, 28]            [2, 4, 28, 28]            48
│    │    └─MaxPool2d: 3-2                    [2, 4, 28, 28]            [2, 4, 14, 14]            --
├─ModuleList: 1-2                             --                        --                        --
│    └─UpBlock2D: 2-2                         [2, 4, 14, 14]            [2, 1, 28, 28]            --
│    │    └─ConvTranspose2d: 3-3              [2, 4, 14, 14]            [2, 4, 28, 28]            68
│    │    └─ConvBlock2D: 3-4                  [2, 4, 28, 28]            [2, 1, 28, 28]

In [ ]:
class ConvBlock3D(nn.Module):
    """Conv -> Activation -> Norm, spatial size unchanged (stride=1)."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm3d(out_channels),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock3D(nn.Module):
    """Encoder step: ConvBlock followed by pooling, which halves depth, height and width."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ConvBlock3D(in_channels, out_channels)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        return self.pool(self.conv(x))


class UpBlock3D(nn.Module):
    """Decoder step: upsampling (linear/trilinear or ConvTranspose) followed by ConvBlock."""
    def __init__(self, in_channels, out_channels, mode="convtranspose"):
        super().__init__()
        if mode == "convtranspose":
            self.upsample = nn.ConvTranspose3d(in_channels, in_channels, kernel_size=2, stride=2)
        elif mode == "linear":
            self.upsample = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        else:
            raise ValueError(f"Unknown upsampling mode: {mode}")
        self.conv = ConvBlock3D(in_channels, out_channels)

    def forward(self, x):
        return self.conv(self.upsample(x))

In [ ]:
class Autoencoder3D(nn.Module):
    """Stacks `depth` DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1, up_mode="convtranspose"):
        super().__init__()
        self.depth = depth
        channels = [in_channels] + [latent_channels] * depth
        self.encoder = nn.ModuleList(
            [DownBlock3D(channels[i], channels[i + 1]) for i in range(depth)]
        )
        self.decoder = nn.ModuleList(
            [UpBlock3D(channels[i + 1], channels[i], mode=up_mode) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))



In [ ]:
class Autoencoder3D(nn.Module):
    """Stacks `depth` DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1, up_mode="convtranspose"):
        super().__init__()
        self.depth = depth
        channels = [in_channels] + [latent_channels] * depth
        self.encoder = nn.ModuleList(
            [DownBlock3D(channels[i], channels[i + 1]) for i in range(depth)]
        )
        self.decoder = nn.ModuleList(
            [UpBlock3D(channels[i + 1], channels[i], mode=up_mode) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))

x3 = torch.randn(2, 1, 32, 32, 32)

ae3 = Autoencoder3D(in_channels=1, latent_channels=4, depth=2)
show_architecture(ae3, x3)
print_compression("3D", depth=2, in_channels=1, latent_channels=4, spatial_dims=3)

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
Autoencoder3D                                 [2, 1, 32, 32, 32]        [2, 1, 32, 32, 32]        --
├─ModuleList: 1-1                             --                        --                        --
│    └─DownBlock3D: 2-1                       [2, 1, 32, 32, 32]        [2, 4, 16, 16, 16]        --
│    │    └─ConvBlock3D: 3-1                  [2, 1, 32, 32, 32]        [2, 4, 32, 32, 32]        120
│    │    └─MaxPool3d: 3-2                    [2, 4, 32, 32, 32]        [2, 4, 16, 16, 16]        --
│    └─DownBlock3D: 2-2                       [2, 4, 16, 16, 16]        [2, 4, 8, 8, 8]           --
│    │    └─ConvBlock3D: 3-3                  [2, 4, 16, 16, 16]        [2, 4, 16, 16, 16]        444
│    │    └─MaxPool3d: 3-4                    [2, 4, 16, 16, 16]        [2, 4, 8, 8, 8]           --
├─ModuleList: 1-2                             --                        --          